# ARC Prize 2026 — ARC-AGI-3 Submission

Built from `agent/my_agent.py` via `scripts/build_notebook.py`. Do not edit cells directly — edit the source file and re-run `make submit`.

In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

In [ ]:
%%writefile /tmp/my_agent.py
"""Phase 1 random reference agent for the ARC-AGI-3 Kaggle submission path.

Follows the ARC-AGI-3-Kaggle-Starter contract (subclass `agents.agent.Agent`, class named `MyAgent`,
`is_done` + `choose_action`). Uses only `latest_frame.available_actions` and the frame state, i.e.
information permitted by the evaluation interface. No learning, no per-game logic.

MODE = "type_uniform": pick uniformly among available action types (ACTION6 = one type), then a uniform
                        (x, y) if ACTION6.
MODE = "flat_uniform" : pick uniformly over the flattened legal space (each simple action + 4096 clicks);
                        clicks dominate whenever ACTION6 is available (plan 5.3).
"""
from __future__ import annotations

import random
from typing import Any

from arcengine import FrameData, GameAction, GameState

from agents.agent import Agent

MODE = "type_uniform"
SEED = 0
GRID = 64
# Provisional: real competition action limit / runtime must be checked on Kaggle (docs/DECISIONS.md).
RANDOM_MAX_ACTIONS = 1000


def sample_action(rng: random.Random, available: list[int], mode: str) -> tuple[int, tuple[int, int] | None]:
    """Return (action_id, (x, y) or None) from the legal set. RESET (0) is never sampled."""
    legal = [a for a in available if a != 0]
    if not legal:
        raise ValueError("no legal actions")
    simple = [a for a in legal if a != 6]
    has_click = 6 in legal
    if mode == "type_uniform":
        a = rng.choice(legal)
    elif mode == "flat_uniform":
        k = rng.randrange(len(simple) + (GRID * GRID if has_click else 0))
        a = simple[k] if k < len(simple) else 6
    else:
        raise ValueError(mode)
    return (a, (rng.randrange(GRID), rng.randrange(GRID))) if a == 6 else (a, None)


class MyAgent(Agent):
    MAX_ACTIONS = RANDOM_MAX_ACTIONS

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        self.mode = MODE
        self.rng = random.Random(f"{SEED}-{self.game_id.split('-')[0]}")

    @property
    def name(self) -> str:
        return f"{super().name}.random-{self.mode}.{self.MAX_ACTIONS}"

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        return latest_frame.state is GameState.WIN

    def choose_action(self, frames: list[FrameData], latest_frame: FrameData) -> GameAction:
        if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
            return GameAction.RESET
        a, xy = sample_action(self.rng, list(latest_frame.available_actions), self.mode)
        action = GameAction.from_id(a)
        if xy is not None:
            action.set_data({"x": xy[0], "y": xy[1]})
        action.reasoning = f"random {self.mode}"
        return action


In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Wait for the gateway sidecar to be ready.
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    # Copy the framework into a writable location.
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    # Drop our agent in as a framework template.
    !cp /tmp/my_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    # Register MyAgent in the framework's agent registry. We rewrite
    # __init__.py because the upstream version eagerly imports
    # templates with deps we don't ship (langgraph, smolagents, etc.).
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write("""from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    'random': Random,
    'myagent': MyAgent,
}
""")

    # Point the framework at the gateway sidecar.
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
""")

    # Run it. The gateway records every action and emits submission.parquet.
    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg \
        python main.py --agent myagent


In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Save-and-run-all (commit) mode: emit a dummy submission so the
    # commit succeeds. The real submission.parquet is produced by the
    # gateway during competition rerun.
    import pandas as pd
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    submission.head()
